# Breakout RL on Colab

This notebook clones the Breakout RL project, installs dependencies, starts training, launches the dashboard service, and exposes it through a temporary public tunnel.

In [ ]:
!nvidia-smi || true

import os
from pathlib import Path

WORKDIR = Path('/content/breakout_rl_server')
REPO = 'https://github.com/0001191/breakout-rl-server.git'

if WORKDIR.exists():
    !rm -rf /content/breakout_rl_server

!git clone {REPO} {WORKDIR}
%cd /content/breakout_rl_server
!pip install -r requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared


In [ ]:
%cd /content/breakout_rl_server
!mkdir -p artifacts/live
!python server_app.py > artifacts/live/server.log 2>&1 &
!python start_training_server.py --total-timesteps 2000000 --device cuda --preview-freq 10000 --status-freq 2000 > artifacts/live/colab_launcher.log 2>&1 &
!sleep 5
!tail -n 20 artifacts/live/server.log || true
!tail -n 20 artifacts/live/train.log || true


In [ ]:
import re
import subprocess
import time

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = proc.stdout.readline()
    if not line:
        continue
    print(line.rstrip())
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print('\nDashboard URL:', public_url + '/dashboard/')
else:
    raise RuntimeError('Failed to obtain cloudflared public URL')


In [ ]:
%cd /content/breakout_rl_server
!tail -n 60 artifacts/live/train.log
